Перед запуском убедитесь, что в корне проекта есть файл .env и в нем заполнены выданные вам креды подключения к базам данных и хранилищу

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, MetaData, Table


In [3]:
# подгружаем .env
load_dotenv()

True

In [4]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [5]:
# Создадим соединения
src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [7]:
# Укажем таблицу для удаления
TABLE_NAME = 'clean_users_churn'

In [7]:
# Функция удаления таблицы с помощью sqlalchemy
def delete_table(table_name, conn):
    metadata = MetaData(bind=conn)
    table = Table(table_name, metadata, autoload=True)
    table.drop(conn)

In [8]:
# Удалим таблицу
delete_table(TABLE_NAME, dst_conn)

/tmp/ipykernel_60323/291150646.py:3: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  metadata = MetaData(bind=conn)


NoSuchTableError: alt_users_churn

In [ ]:
# Проверим, что таблицы больше нет
pd.read_sql(f'select * from {TABLE_NAME}', dst_conn)

In [15]:
from airflow.providers.postgres.hooks.postgres import PostgresHook

def create_table(**kwargs):
    from sqlalchemy import MetaData, Table, Column, String, Integer, DateTime, \
        Float, inspect, UniqueConstraint
    engine = PostgresHook(dst_db).get_sqlalchemy_engine()
    metadata = MetaData()
    uc_table = Table(
        "alt_users_churn",
        metadata,
        Column("id", Integer, primary_key=True, autoincrement=True),
        Column("customer_id", String),
        Column("begin_date", DateTime),
        Column("end_date", DateTime),
        Column("type", String),
        Column("paperless_billing", String),
        Column("payment_method", String),
        Column("monthly_charges", Float),
        Column("total_charges", Float),
        Column("internet_service", String),
        Column("online_security", String),
        Column("online_backup", String),
        Column("device_protection", String),
        Column("tech_support", String),
        Column("streaming_tv", String),
        Column("streaming_movies", String),
        Column("gender", String),
        Column("senior_citizen", Integer),
        Column("partner", String),
        Column("dependents", String),
        Column("multiple_lines", String),
        Column("target", Integer),
        UniqueConstraint('customer_id', name='unique_customer_id_constraint')
    )
    if not inspect(engine).has_table(uc_table.name): 
        metadata.create_all(engine)

In [16]:
create_table()

[2026-09-24T18:43:01.037+0000] {connection.py:471} ERROR - Unable to retrieve connection from secrets backend (MetastoreBackend). Checking subsequent secrets backend.
Traceback (most recent call last):
  File "/home/mle-user/mle_projects/mle-airflow/.venv_mle_airflow/lib/python3.10/site-packages/sqlalchemy/engine/base.py", line 1910, in _execute_context
    self.dialect.do_execute(
  File "/home/mle-user/mle_projects/mle-airflow/.venv_mle_airflow/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 736, in do_execute
    cursor.execute(statement, parameters)
sqlite3.OperationalError: no such table: connection

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/mle-user/mle_projects/mle-airflow/.venv_mle_airflow/lib/python3.10/site-packages/airflow/models/connection.py", line 466, in get_connection_from_secrets
    conn = secrets_backend.get_connection(conn_id=conn_id)
  File "/home/mle-user/mle_projects/mle

AirflowNotFoundException: The conn_id `playground_mle_20260913_c5f49a0b90` isn't defined

In [8]:
# Закроем соединения в конце работы
src_conn.dispose()
dst_conn.dispose()